# R3: AI public discourse content shifts over time within a stable spread

**Finding being tested**: the spread of AI concerns across the cluster space
is *stable* across two decades; the content mix *shifts* in interpretable
ways that reflect both substantive evolution and the changing landscape of
dialogue commissioning.

**Two-part structure of the finding**:

1. **Stable spread** — normalised entropy of AI concern distributions stays
   at ~0.91–0.95 across all four time windows. AI concern discourse has
   always been broadly distributed, not dominated by one or two themes.

2. **Shifting content mix** — while the spread is stable, the *relative*
   prominence of specific clusters changes across periods. Data themes
   dominated in 2004–2017; governance and equity rose in 2018–2020; data
   themes returned strongly in 2021–2023; exclusion and oversight joined
   data privacy at the top in 2024–2025.

**Four time windows** (chosen to ensure adequate sample size per window):
- 2004–2017: 1,432 concern phrases, n=5 documents (early health-data dialogues)
- 2018–2020: 1,568 phrases, n=6 (general-attitudes and ethical AI work)
- 2021–2023: 2,427 phrases, n=19 (post-pandemic biometrics, trust dialogues)
- 2024–2025: 1,528 phrases, n=11 (current AI policy moment)

**Caveat**: temporal patterns reflect in part what *kinds* of dialogues were
commissioned in each period, not just shifting public attitudes.

This notebook loads pre-computed artifacts from `01_processing.ipynb` and
`01a_clustering.ipynb`. No API calls are made here.

In [ ]:
# @title Load pre-computed artifacts
# Run this cell before any analysis cell.  It loads all outputs written by
# 01_processing.ipynb from disk so this notebook never calls the OpenAI API
# or re-runs k-means.
from pathlib import Path
import pub_dialogue.utils as du
from pub_dialogue.utils import show_status, show_complete, AccessStage, AddressStage
import pandas as pd
import numpy as np

# --- CIP-0010: stage objects centralise all path / parameter constants ---
_access  = AccessStage()
_address = AddressStage(access=_access)
OUTPUT_FOLDER     = _access.output_folder
CHECKPOINT_FOLDER = _access.checkpoint_folder
TECH_COL          = _address.tech_col

a = _access.load_artifacts()

chunks_df    = a["chunks_df"]
concerns_df  = a["concerns_df"]
benefits_df  = a["benefits_df"]

concern_embeddings  = a["concern_embeddings"]
benefit_embeddings  = a["benefit_embeddings"]
concern_centroids   = a["concern_centroids"]
benefit_centroids   = a["benefit_centroids"]

concern_ids          = a["concern_ids"]
benefit_ids          = a["benefit_ids"]
cluster_labels       = a["cluster_labels"]
benefit_cluster_labels = a["benefit_cluster_labels"]
cluster_summary_df   = a["cluster_summary_df"]
benefit_cluster_summary_df = a["benefit_cluster_summary_df"]

framing_lens_mappings         = a["framing_lens_mappings"]
benefit_framing_lens_mappings = a["benefit_framing_lens_mappings"]

cluster_entropy           = a["cluster_entropy"]
cluster_entropy_norm      = a["cluster_entropy_norm"]
cross_cutting_clusters    = a["cross_cutting_clusters"]

benefit_cluster_entropy          = a["benefit_cluster_entropy"]
normalized_entropy_benefits      = a["normalized_entropy_benefits"]
cross_cutting_clusters_benefits  = a["cross_cutting_clusters_benefits"]

# Convenience: CLUSTER_LABELS / BENEFIT_CLUSTER_LABELS dicts used by plots
CLUSTER_LABELS        = {int(k): v for k, v in cluster_labels.items()}
BENEFIT_CLUSTER_LABELS = {int(k): v for k, v in benefit_cluster_labels.items()}

# Re-apply technology_meta merge if not already present in the loaded CSV
if "technology_meta" not in concerns_df.columns:
    import pandas as pd
    _tech = chunks_df[["chunk_id", "technology_meta"]]
    concerns_df = concerns_df.merge(_tech, on="chunk_id", how="left")
    benefits_df = benefits_df.merge(_tech, on="chunk_id", how="left")

print(f"Artifacts loaded from {OUTPUT_FOLDER} / {CHECKPOINT_FOLDER}")
print(f"  Chunks: {len(chunks_df):,}  |  Concerns: {len(concerns_df):,}  |  Benefits: {len(benefits_df):,}")

## Metrics for measuring concentration and spread

Three complementary metrics are used to characterise temporal distributions:
- **Normalised entropy**: the primary spread measure (0 = all one cluster,
  1 = perfectly uniform). The R3 stability claim rests on this metric
  remaining at ~0.91–0.95 across all four time windows.
- **HHI (Herfindahl-Hirschman Index)**: an alternative concentration measure;
  used as a robustness check on the entropy result.
- **Top-k share**: the combined share of the top k clusters; provides an
  intuitive "how concentrated?" answer alongside entropy.

In [ ]:
# normalized_entropy, hhi, topk_share, parse_year — imported from dialogue_utils (see Cell 5)
pass

## Stable spread: entropy across four time windows

Year-by-year entropy calculations are noisy because some years have very few
documents (notably 2020 has only 31 chunks from one fragmentary document).
The four time windows above pool years to achieve adequate sample sizes.

**The stability result**: normalised entropy stays at ~0.91–0.95 across all
four windows. This is a high-entropy regime — AI concern discourse has
always been broadly distributed across the cluster space. The finding that
entropy is *stable* rules out a simple "evolution" story in which AI
concerns became more concentrated or more diffuse over time.

**The 2020 gap**: 2020 has only one document (31 chunks). Year-by-year
plots exclude or flag this year; it does not distort the window-level
analysis because it falls within the 2018–2020 window where n=6 documents
give adequate coverage of the period.

In [ ]:
# @title (v16) Wider-time-window AI concern analysis
# Issue 4 partial fix from the audit: year-by-year AI concern data is
# extremely sparse for some years (e.g. 2024 had 9 phrases in v15). With
# small N, normalised entropy is mathematically forced toward its maximum
# regardless of the underlying distribution, confounding any temporal-trend
# claim about "concern diversity" with sample size.
#
# This cell repeats the temporal AI analysis using 4 wider windows
# (2004-2017, 2018-2020, 2021-2023, 2024-2025) so each window contains
# enough data to make the entropy/share numbers interpretable. The
# year-by-year analysis is retained above; treat this as the primary table
# for any quantitative temporal claim in the paper.

show_status("Running wider-window temporal AI analysis...")

from scipy.stats import entropy
from pub_dialogue.address import assign_window

ai_concerns_window = concerns_df[concerns_df["technology_meta"] == "AI"].copy()
ai_concerns_window["time_window"] = ai_concerns_window["year"].apply(assign_window)
ai_concerns_window = ai_concerns_window.dropna(subset=["time_window"])

# Per-window: total phrases, cluster distribution, entropy
window_summary = []
for window, group in ai_concerns_window.groupby("time_window"):
    n_phrases = len(group)
    cluster_counts = group["cluster_id"].value_counts()
    probs = (cluster_counts / cluster_counts.sum()).values if cluster_counts.sum() > 0 else np.array([])
    raw_ent = float(entropy(probs)) if len(probs) > 1 else 0.0
    norm_ent = raw_ent / np.log(len(cluster_counts)) if len(cluster_counts) > 1 else 0.0
    window_summary.append({
        "time_window": window,
        "n_phrases": n_phrases,
        "n_clusters_present": int((cluster_counts > 0).sum()),
        "n_documents": int(group["source_file"].nunique()),
        "raw_entropy": raw_ent,
        "normalized_entropy_within_window": norm_ent,
    })

window_summary_df = pd.DataFrame(window_summary).sort_values("time_window")
window_summary_df.to_csv(OUTPUT_FOLDER / "ai_concern_window_summary.csv", index=False)

print("Wider-window AI concern summary:")
print(window_summary_df.to_string(index=False))
print()
print("Note: 'normalized_entropy_within_window' is normalised by the number of")
print("clusters that contain any phrases in that window — so it is bounded by")
print("the data, not by the absolute cluster count. With more data per window,")
print("these values are more interpretable than the year-by-year ones above.")

# Also: per-window top clusters (qualitative)
top_clusters_per_window = []
for window, group in ai_concerns_window.groupby("time_window"):
    counts = group["cluster_id"].value_counts().head(10)
    for cid, n in counts.items():
        label = CLUSTER_LABELS.get(cid, f"Cluster {cid}") if "CLUSTER_LABELS" in globals() else f"Cluster {cid}"
        top_clusters_per_window.append({
            "time_window": window,
            "cluster_id": cid,
            "label": label,
            "n_phrases": int(n),
            "share_within_window": float(n) / len(group),
        })

top_window_df = pd.DataFrame(top_clusters_per_window)
top_window_df.to_csv(OUTPUT_FOLDER / "ai_concern_window_top_clusters.csv", index=False)
print(f"\nWrote top-10-clusters-per-window table to ai_concern_window_top_clusters.csv")

show_complete("Wider-window analysis complete")

## Compute AI concern PCA trajectory

Projects each year's concern distribution into 2D using PCA on the
year × cluster matrix. The trajectory shows how the centre of gravity of
AI concern discourse has moved over time, independently of whether the
spread (entropy) has changed.

In [ ]:
# @title Compute AI concern PCA trajectory (traj)
# traj: rows = years, columns = year / pc1 / pc2
# CIP-0010: delegated to _address.concern_trajectory()

traj = _address.concern_trajectory(concerns_df, concern_embeddings, concern_ids)
print(f"traj: {len(traj)} years")


## Visualise AI concern trajectory over time (2D displacement plot)

The trajectory is centred on the first year so displacement indicates
movement away from the initial concern profile. Rapid movement indicates
a large shift in which clusters dominate; slow movement indicates stability.

Note: PCA trajectory and entropy are complementary — high entropy means
spread across many clusters; large displacement means the *centre* has
moved, even if spread is stable.

In [ ]:
# @title Visualise AI concern trajectory over time

import matplotlib.pyplot as plt
import numpy as np

# Use traj DataFrame from the previous cell
# columns: year, pc1, pc2

traj2 = traj.sort_values("year").reset_index(drop=True)

# Center trajectory on first year
x0, y0 = traj2.loc[0, ["pc1", "pc2"]]
dx = traj2["pc1"] - x0
dy = traj2["pc2"] - y0

years = traj2["year"].to_numpy()

plt.figure(figsize=(6, 6))
sc = plt.scatter(dx, dy, c=years, cmap="viridis", s=80)
plt.plot(dx, dy, alpha=0.6)

plt.axhline(0, color="grey", linewidth=1, alpha=0.5)
plt.axvline(0, color="grey", linewidth=1, alpha=0.5)

plt.xlabel("Displacement in concern space (PC1)")
plt.ylabel("Displacement in concern space (PC2)")
plt.title("AI concern profile over time\n(displacement from initial position)")

cbar = plt.colorbar(sc)
cbar.set_label("Year")

plt.tight_layout()
plt.show()


## Compute AI concern year × cluster matrix

Builds the year × cluster salience matrix for AI documents: each cell is
the share of AI concern phrases in that year that belong to that cluster.
This matrix feeds into the heatmap (Figure 4) and the rising/declining
cluster analysis.

In [ ]:
# @title Compute AI concern year × cluster matrix (ai_year)
# ai_year: rows = years, columns = concern cluster IDs,
# values = FRACTION OF DOCUMENTS in that year mentioning that cluster.
# Methodology: CIP-0009 Approach B (document-level binary weighting).
# CIP-0010: delegated to _address.concern_year_matrix()

ai_year = _address.concern_year_matrix(concerns_df, chunks_df)
years = sorted(ai_year.index.tolist())

print(f"ai_year: {ai_year.shape}  (years × concern clusters for AI, fraction of docs)")
print(ai_year.round(3))


## Rising and declining concern clusters (linear trend slopes)

Applies linear regression to each cluster's year-normalised salience time
series to identify which clusters are systematically gaining or losing
attention across the corpus years. This provides a more formal summary of
the content-mix shift claim in R3 than the four-window snapshot alone.

**Defensible claim**: trends should be interpreted cautiously given the
uneven corpus (few documents before 2018; the 2020 gap). The window-level
analysis in Figure 4 is the primary evidence; rising/declining slopes are
supplementary.

In [ ]:
# @title Analyse AI concern salience trajectories
# Shows which concern clusters rise or fall most within AI discourse over time.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ai_year and years already exist from previous cells
# ai_year: years × clusters (raw salience)

# Normalize within each year so values sum to 1 (relative attention)
ai_rel = ai_year.div(ai_year.sum(axis=1), axis=0).fillna(0)

# Compute linear trend (slope) for each cluster
# Using simple OLS slope over time
t = np.arange(len(ai_rel))  # time index (no assumption of equal year gaps)
slopes = {}
for c in ai_rel.columns:
    y = ai_rel[c].to_numpy()
    if np.all(y == 0):
        slopes[c] = np.nan
    else:
        # slope of y ~ t
        slopes[c] = np.polyfit(t, y, 1)[0]

trend = pd.Series(slopes).dropna().sort_values(ascending=False)

# Select top rising and declining clusters
N = 8
top_up = trend.head(N)
top_down = trend.tail(N)

# Plot trajectories
plt.figure(figsize=(11, 5))

# Rising
plt.subplot(1, 2, 1)
for c in top_up.index:
    plt.plot(ai_rel.index, ai_rel[c], marker="o", label=c)
plt.title("AI concerns increasing in relative salience")
plt.xlabel("Year")
plt.ylabel("Share of AI attention")
plt.legend(fontsize=9)

# Declining
plt.subplot(1, 2, 2)
for c in top_down.index:
    plt.plot(ai_rel.index, ai_rel[c], marker="o", label=c)
plt.title("AI concerns decreasing in relative salience")
plt.xlabel("Year")
plt.ylabel("Share of AI attention")
plt.legend(fontsize=9)

plt.tight_layout()
plt.show()

display(
    pd.DataFrame({
        "slope": trend
    }).assign(direction=lambda d: np.where(d["slope"] > 0, "rising", "declining"))
)


## AI concern salience heatmap (year × cluster, top clusters only)

Year-by-year version of Figure 4, using individual years rather than time
windows. Rows are ordered by which year each cluster peaked in, producing
a diagonal pattern when the content mix shifts systematically.

The **stable spread + shifting content** story is readable here: each year
has a distinct top-cluster profile (content shifts), but no single cluster
dominates the year (stable spread). The 2020 column is sparse due to the
single-document year.

In [ ]:
# @title Visualise AI concern salience over time
# Rows = concern clusters
# Columns = years
# Values = relative salience within AI discourse (row-normalised per year)
#
# ai_year holds document-weighted fractions (CIP-0009 Approach B):
# each cell = fraction of AI documents in that year mentioning that cluster.
# We then re-normalise within each year to get the RELATIVE share of attention.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ai_year: years × clusters (doc-fraction per year, from temporal_cluster_frequency)
# years is a sorted array of years

# 1) Normalise within each year (relative attention)
ai_rel = ai_year.div(ai_year.sum(axis=1), axis=0).fillna(0)

# 2) Select clusters to display
# Option A: top N by overall AI salience
N = 20
top_clusters = ai_rel.sum(axis=0).sort_values(ascending=False).head(N).index

heat = ai_rel[top_clusters]

# 3) Order clusters by when they peak (helps visual interpretation)
peak_year_idx = heat.values.argmax(axis=0)
order = np.argsort(peak_year_idx)
heat = heat.iloc[:, order]

# Transpose so clusters are rows
heat_T = heat.T

# 4) Plot heat map
plt.figure(figsize=(12, 7))
im = plt.imshow(
    heat_T,
    aspect="auto",
    cmap="viridis"
)

plt.colorbar(im, label="Share of AI attention (within year)")

plt.yticks(
    ticks=np.arange(len(heat_T.index)),
    labels=heat_T.index
)
plt.xticks(
    ticks=np.arange(len(heat_T.columns)),
    labels=heat_T.columns,
    rotation=45
)

plt.xlabel("Year")
plt.ylabel("Concern cluster")
plt.title("AI public concerns over time\n(relative salience within AI discourse)")

plt.tight_layout()
plt.show()


## Benefits side: temporal dynamics

The same temporal analysis repeated for benefit phrases. The benefit
trajectory and salience heatmap are supplementary to the concern-side
analysis — confirming that the stable-spread / shifting-content pattern
holds on the benefit side as well, and identifying which benefit themes
are rising (e.g. personalisation, research quality) vs declining
(e.g. environmental and local economic benefits).

In [ ]:
# normalized_entropy, hhi, topk_share, parse_year — imported from dialogue_utils (see Cell 5)
pass

## Compute AI benefit PCA trajectory

In [ ]:
# @title Compute AI benefit PCA trajectory (benefit_traj)
# benefit_traj: rows = years, columns = year / pc1 / pc2
# CIP-0010: delegated to _address.benefit_trajectory()

benefit_traj = _address.benefit_trajectory(benefits_df, benefit_embeddings, benefit_ids)
print(f"benefit_traj: {len(benefit_traj)} years")


## Visualise AI benefit trajectory over time

In [ ]:
# @title Visualise AI benefit trajectory over time

import matplotlib.pyplot as plt
import numpy as np

benefit_traj2 = benefit_traj.sort_values("year").reset_index(drop=True)

# Center trajectory on first year
x0, y0 = benefit_traj2.loc[0, ["pc1", "pc2"]]
dx = benefit_traj2["pc1"] - x0
dy = benefit_traj2["pc2"] - y0

benefit_years_arr = benefit_traj2["year"].to_numpy()

plt.figure(figsize=(6, 6))
sc = plt.scatter(dx, dy, c=benefit_years_arr, cmap="viridis", s=80)
plt.plot(dx, dy, alpha=0.6)

plt.axhline(0, color="grey", linewidth=1, alpha=0.5)
plt.axvline(0, color="grey", linewidth=1, alpha=0.5)

plt.xlabel("Displacement in benefit space (PC1)")
plt.ylabel("Displacement in benefit space (PC2)")
plt.title("AI benefit profile over time\n(displacement from initial position)")

cbar = plt.colorbar(sc)
cbar.set_label("Year")

plt.tight_layout()
plt.show()


## Compute AI benefit year × cluster matrix

In [ ]:
# @title Compute AI benefit year × cluster matrix (benefit_ai_year)
# benefit_ai_year: rows = years, columns = benefit cluster IDs,
# values = FRACTION OF DOCUMENTS in that year mentioning that cluster.
# Same document-level binary weighting as ai_year (CIP-0009 Approach B).
# CIP-0010: delegated to _address.benefit_year_matrix()

benefit_ai_year = _address.benefit_year_matrix(benefits_df, chunks_df)
benefit_years = sorted(benefit_ai_year.index.tolist())

print(f"benefit_ai_year: {benefit_ai_year.shape}  (years × benefit clusters for AI, fraction of docs)")
print(benefit_ai_year.round(3))


## Rising and declining benefit clusters (linear trend slopes)

In [ ]:
# @title Analyse AI benefit salience trajectories

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

benefit_ai_rel = benefit_ai_year.div(benefit_ai_year.sum(axis=1), axis=0).fillna(0)

# Compute linear trend (slope) for each cluster
# Note: this uses a year-index, not the actual years, so non-uniform gaps
# are treated as uniform. See methodology limitations.
t = np.arange(len(benefit_ai_rel))
slopes = {}
for c in benefit_ai_rel.columns:
    y = benefit_ai_rel[c].to_numpy()
    if np.all(y == 0):
        slopes[c] = np.nan
    else:
        slopes[c] = np.polyfit(t, y, 1)[0]

trend = pd.Series(slopes).dropna().sort_values(ascending=False)

# Select top rising and declining clusters
N = 8
top_up = trend.head(N)
top_down = trend.tail(N)

# Plot trajectories
plt.figure(figsize=(11, 5))

# Rising
plt.subplot(1, 2, 1)
for c in top_up.index:
    plt.plot(benefit_ai_rel.index, benefit_ai_rel[c], marker="o", label=c)
plt.title("AI benefits increasing in relative salience")
plt.xlabel("Year")
plt.ylabel("Share of AI attention")
plt.legend(fontsize=9)

# Declining
plt.subplot(1, 2, 2)
for c in top_down.index:
    plt.plot(benefit_ai_rel.index, benefit_ai_rel[c], marker="o", label=c)
plt.title("AI benefits decreasing in relative salience")
plt.xlabel("Year")
plt.ylabel("Share of AI attention")
plt.legend(fontsize=9)

plt.tight_layout()
plt.show()

display(
    pd.DataFrame({
        "slope": trend
    }).assign(direction=lambda d: np.where(d["slope"] > 0, "rising", "declining"))
)


## AI benefit salience heatmap (year × cluster, top clusters only)

In [ ]:
# @title Visualise AI benefit salience over time
# Rows = benefit clusters
# Columns = years
# Values = relative salience within AI discourse (row-normalised per year)
#
# benefit_ai_year holds document-weighted fractions (CIP-0009 Approach B).
# Re-normalised within each year to get the relative share of attention.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1) Normalise within each year (relative attention)
benefit_ai_rel = benefit_ai_year.div(benefit_ai_year.sum(axis=1), axis=0).fillna(0)

# 2) Select top N clusters by overall AI salience
N = 20
top_clusters = benefit_ai_rel.sum(axis=0).sort_values(ascending=False).head(N).index

heat = benefit_ai_rel[top_clusters]

# 3) Order clusters by when they peak
peak_year_idx = heat.values.argmax(axis=0)
order = np.argsort(peak_year_idx)
heat = heat.iloc[:, order]

heat_T = heat.T

plt.figure(figsize=(12, 7))
im = plt.imshow(
    heat_T,
    aspect="auto",
    cmap="viridis"
)

plt.colorbar(im, label="Share of AI attention (within year)")

plt.yticks(
    ticks=np.arange(len(heat_T.index)),
    labels=heat_T.index
)
plt.xticks(
    ticks=np.arange(len(heat_T.columns)),
    labels=heat_T.columns,
    rotation=45
)

plt.xlabel("Year")
plt.ylabel("Benefit cluster")
plt.title("AI public benefits over time\n(relative salience within AI discourse)")

plt.tight_layout()
plt.show()


## Figure 4: AI concern profile across four time windows (paper asset)

Heatmap of cluster shares across the four time windows, for the union of
clusters appearing in the top 10 of any window. Rows are ordered by which
window each cluster peaks in, producing a diagonal pattern that visualises
the content shift.

**Reading the chart**:
- **Diagonal pattern** = content mix shifts systematically across windows.
- **No single row dominates all columns** = spread is stable (high entropy
  in every window).
- The data-theme clusters (Misuse of personal data, Privacy and data
  security) are visible across all windows but peak in 2004–2017 and
  2021–2023. Governance and equity themes are relatively more prominent
  in 2018–2020 and 2024–2025.

This is Figure 4 of the paper.

**Defensible claim**: "AI public discourse has consistently centred on data
themes, with their relative prominence ebbing and flowing as other concerns
(governance, equity, exclusion) come into focus alongside them."

In [ ]:
# @title (paper) Figure 4: AI concern profile across four time windows
# Heatmap of cluster shares within each time window for the union of clusters
# that appeared in the top 10 of any window. Rows are ordered by which
# window each cluster reaches its peak in, so the diagonal pattern reads
# as the "content shifts within stable spread" finding.

import json as _json
import numpy as _np
import matplotlib.pyplot as _plt
import matplotlib.colors as _mcolors

PAPER_ASSETS = OUTPUT_FOLDER / "paper_assets"
PAPER_ASSETS.mkdir(exist_ok=True)

# Load traceability and the existing top-clusters-per-window file
_trace = pd.read_csv(OUTPUT_FOLDER / "traceability_paragraphs.csv")
_top_window = pd.read_csv(OUTPUT_FOLDER / "ai_concern_window_top_clusters.csv")
_summary = pd.read_csv(OUTPUT_FOLDER / "cluster_summary.csv")

from pub_dialogue.address import _parse_listcol, assign_window as _assign_window

_ai = _trace[_trace["technology_meta"] == "AI"].copy()
_ai["cluster_ids_parsed"] = _ai["cluster_ids"].apply(_parse_listcol)
_ai["window"] = _ai["year_meta"].apply(_assign_window)

_exploded = _ai[["window", "cluster_ids_parsed"]].explode("cluster_ids_parsed").dropna()
_exploded["cluster_id"] = pd.to_numeric(_exploded["cluster_ids_parsed"], errors="coerce")
_exploded = _exploded.dropna(subset=["cluster_id"])
_exploded["cluster_id"] = _exploded["cluster_id"].astype(int)

_counts = _exploded.groupby(["window", "cluster_id"]).size().reset_index(name="n")
_window_totals = _exploded.groupby("window").size().reset_index(name="window_total")
_counts = _counts.merge(_window_totals, on="window")
_counts["share_within_window"] = _counts["n"] / _counts["window_total"]

_union = _top_window["cluster_id"].unique()
_hm = _counts[_counts["cluster_id"].isin(_union)].copy()

_window_order = ["2004-2017", "2018-2020", "2021-2023", "2024-2025"]
_pivot = _hm.pivot(index="cluster_id", columns="window", values="share_within_window").fillna(0)
_pivot = _pivot[_window_order]

_labels_map = dict(zip(_summary["cluster_id"], _summary["label"]))
_pivot["label"] = _pivot.index.map(_labels_map)

def _peak_window(row):
    shares = row[_window_order]
    return _window_order.index(shares.idxmax())

_pivot["peak_window"] = _pivot.apply(_peak_window, axis=1)
_pivot["peak_share"] = _pivot[_window_order].max(axis=1)
_pivot = _pivot.sort_values(["peak_window", "peak_share"], ascending=[True, False])

# Save the underlying data
_pivot[_window_order + ["label"]].to_csv(PAPER_ASSETS / "figure4_data.csv")

# Render
_data = _pivot[_window_order].values * 100
_labels = _pivot["label"].tolist()
_n = len(_labels)

fig, ax = _plt.subplots(figsize=(8.5, max(7, _n * 0.35)))
_vmax = _np.percentile(_data[_data > 0], 95)
_im = ax.imshow(_data, aspect="auto", cmap=_plt.get_cmap("Blues"),
                norm=_mcolors.Normalize(vmin=0, vmax=_vmax))

ax.set_xticks(range(len(_window_order)))
ax.set_xticklabels(_window_order, fontsize=10)
ax.set_yticks(range(_n))
ax.set_yticklabels(_labels, fontsize=9)
ax.tick_params(axis="x", top=True, bottom=False, labeltop=True, labelbottom=False)

for i in range(_n):
    for j in range(len(_window_order)):
        v = _data[i, j]
        if v == 0:
            ax.text(j, i, "·", ha="center", va="center", fontsize=8.5, color="#bbbbbb")
        else:
            color = "white" if (v / _vmax) > 0.55 else "#222222"
            ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=8.5, color=color)
        if v > _vmax:
            ax.text(j + 0.35, i - 0.35, "▲", ha="center", va="center", fontsize=7, color="white")

for i in range(_n + 1):
    ax.axhline(i - 0.5, color="white", linewidth=0.6)
for j in range(len(_window_order) + 1):
    ax.axvline(j - 0.5, color="white", linewidth=0.6)

cbar = _plt.colorbar(_im, ax=ax, fraction=0.025, pad=0.02)
cbar.set_label("Share of AI concern phrases in window (%)", fontsize=9)
cbar.ax.tick_params(labelsize=8)

ax.set_title(f"AI concern profile across four time windows\n"
             f"(union of top-10 clusters from each window; n={_n} clusters)",
             fontsize=11, pad=12)

fig.text(0.02, 0.01,
         "Rows ordered by the time window in which each cluster reaches its peak share. "
         "Cells show share of AI concern phrases within window. · = no phrases. "
         f"▲ = above colour-scale cap of {_vmax:.1f}%.",
         fontsize=7.5, color="#555555", ha="left")

_plt.tight_layout(rect=[0, 0.04, 1, 1])
_plt.savefig(PAPER_ASSETS / "figure4_ai_concerns_across_windows.png",
             dpi=200, bbox_inches="tight", facecolor="white")
_plt.show()
print(f"Saved: {PAPER_ASSETS / 'figure4_ai_concerns_across_windows.png'}")